# Exploratory Data Analysis: Italian Public Procurement (2006-2021)
## 12.1M Records | 651,793 Unique Contracts | Network Analysis + Governance Risk + Anomaly Detection

In [ ]:
import duckdb
import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = 'data/raw/IT_DIB_2023.parquet'
PNG_DIR = Path('reports/eda_outputs')
PNG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name: str, title: str = ''):
    filepath = PNG_DIR / f'{name}.png'
    fig.write_image(filepath, width=1200, height=700, scale=2)
    print(f'✓ Saved: {name}.png')

print(f'✓ Setup complete. Output: {PNG_DIR}')

In [ ]:
df_raw = pl.read_parquet(PARQUET_PATH)
print(f'✓ Loaded {len(df_raw):,} records | {len(df_raw.columns)} columns')

In [ ]:
# FIGURE 1: Data Quality
null_counts = []
for col in df_raw.columns:
    null_count = df_raw[col].null_count()
    if null_count > 0:
        null_counts.append({'column': col, 'nulls': null_count, 'pct': 100*null_count/len(df_raw)})

null_df = pl.DataFrame(null_counts).sort('nulls', descending=True).head(15)

fig1 = px.bar(null_df.to_pandas(), x='column', y='nulls', 
              title='Figure 1: Data Quality - Missing Values',
              color='pct', color_continuous_scale='Reds')
fig1.update_layout(height=700, xaxis_tickangle=-45)
fig1.show()
save_fig(fig1, '01_schema_nulls', 'Data Quality')

In [ ]:
# FIGURE 2: Temporal Coverage
yearly = duckdb.sql(f"""
    SELECT YEAR(tender_awarddecisiondate) as year, COUNT(*) as count
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_awarddecisiondate IS NOT NULL
    GROUP BY year ORDER BY year
""").pl()

fig2 = px.line(yearly.to_pandas(), x='year', y='count', markers=True,
              title='Figure 2: Temporal Coverage - Contract Awards by Year')
fig2.update_layout(height=700)
fig2.show()
save_fig(fig2, '02_temporal_coverage', 'Temporal Coverage')

In [ ]:
# FIGURE 3: Buyer Value Distribution
price_sample = duckdb.sql(f"""
    SELECT tender_finalprice FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_finalprice BETWEEN 10000 AND 5000000 LIMIT 50000
""").pl()

fig3 = px.box(y=price_sample['tender_finalprice'],
             title='Figure 3: Buyer Ecosystem - Contract Value Distribution')
fig3.update_layout(height=700, showlegend=False)
fig3.update_yaxes(type='log')
fig3.show()
save_fig(fig3, '03_buyer_value_distribution', 'Value Distribution')

In [ ]:
# FIGURE 4: Buyer Dependency Ratio
buyer_dep = duckdb.sql(f"""
    WITH buyer_supplier AS (
        SELECT tender_buyersme_id as buyer_id, tender_suppliersme_id as supplier_id,
               SUM(COALESCE(tender_finalprice, 0)) as supplier_value,
               ROW_NUMBER() OVER (PARTITION BY tender_buyersme_id ORDER BY SUM(COALESCE(tender_finalprice, 0)) DESC) as rank
        FROM read_parquet('{PARQUET_PATH}')
        WHERE tender_buyersme_id IS NOT NULL AND tender_suppliersme_id IS NOT NULL
        GROUP BY buyer_id, supplier_id
    ),
    top5 AS (SELECT buyer_id, SUM(supplier_value) as top5_value FROM buyer_supplier WHERE rank <= 5 GROUP BY buyer_id),
    total AS (SELECT buyer_id, SUM(supplier_value) as total_value FROM buyer_supplier GROUP BY buyer_id)
    SELECT ROUND(top5_value / NULLIF(total_value, 0), 2) as dependency_ratio
    FROM top5 JOIN total USING(buyer_id) WHERE total_value > 0
""").pl()

fig4 = px.histogram(buyer_dep.to_pandas(), x='dependency_ratio', nbins=50,
                   title='Figure 4: Buyer Dependency Ratio - Top-5 Supplier Concentration')
fig4.update_layout(height=700)
fig4.add_vline(x=0.5, line_dash='dash', line_color='red')
fig4.show()
save_fig(fig4, '04_buyer_dependency_distribution', 'Dependency Ratio')

In [ ]:
# FIGURE 5: Top Suppliers
top_suppliers = duckdb.sql(f"""
    SELECT tender_suppliersme_id as supplier_id, COUNT(*) as contract_count,
           SUM(COALESCE(tender_finalprice, 0)) as total_value
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_suppliersme_id IS NOT NULL
    GROUP BY supplier_id ORDER BY total_value DESC LIMIT 20
""").pl()

fig5 = px.bar(top_suppliers.to_pandas(), x='supplier_id', y='total_value',
             title='Figure 5: Top 20 Suppliers by Contract Value',
             color='contract_count', color_continuous_scale='Blues')
fig5.update_layout(height=700, xaxis_tickangle=-45)
fig5.show()
save_fig(fig5, '05_top_suppliers', 'Top Suppliers')

In [ ]:
# FIGURE 6: Supplier Trend
supplier_trend = duckdb.sql(f"""
    SELECT YEAR(tender_awarddecisiondate) as year,
           COUNT(DISTINCT tender_suppliersme_id) as supplier_count
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_awarddecisiondate IS NOT NULL
    GROUP BY year ORDER BY year
""").pl()

fig6 = px.line(supplier_trend.to_pandas(), x='year', y='supplier_count', markers=True,
              title='Figure 6: Supplier Ecosystem - Active Suppliers per Year')
fig6.update_layout(height=700)
fig6.show()
save_fig(fig6, '06_supplier_churn', 'Supplier Trend')

In [ ]:
# FIGURE 7: Procedure Types
procedures = duckdb.sql(f"""
    SELECT tender_proceduretype as procedure, COUNT(*) as count
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_proceduretype IS NOT NULL
    GROUP BY procedure ORDER BY count DESC
""").pl()

fig7 = px.pie(procedures.to_pandas(), values='count', names='procedure',
             title='Figure 7: Contract Distribution by Procedure Type')
fig7.update_layout(height=700)
fig7.show()
save_fig(fig7, '07_procedure_distribution', 'Procedure Distribution')

In [ ]:
# FIGURE 8: Single-Bidder Rate
single_bid = duckdb.sql(f"""
    SELECT tender_proceduretype as procedure, COUNT(*) as total,
           SUM(CASE WHEN lot_bidscount = 1 THEN 1 ELSE 0 END) as single_bid_count,
           100.0 * SUM(CASE WHEN lot_bidscount = 1 THEN 1 ELSE 0 END) / COUNT(*) as single_bid_pct
    FROM read_parquet('{PARQUET_PATH}')
    WHERE lot_bidscount IS NOT NULL AND tender_proceduretype IS NOT NULL
    GROUP BY procedure ORDER BY single_bid_pct DESC
""").pl()

fig8 = px.bar(single_bid.to_pandas(), x='procedure', y='single_bid_pct',
             title='Figure 8: Single-Bidder Rate by Procedure (Governance Risk)',
             color='single_bid_pct', color_continuous_scale='RdYlGn_r')
fig8.update_layout(height=700, xaxis_tickangle=-45)
fig8.show()
save_fig(fig8, '08_single_bidder_by_procedure', 'Single-Bidder Rate')

In [ ]:
# FIGURE 9: Bid Distribution
bids = duckdb.sql(f"""
    SELECT lot_bidscount as bid_count, COUNT(*) as contract_count
    FROM read_parquet('{PARQUET_PATH}')
    WHERE lot_bidscount IS NOT NULL AND lot_bidscount <= 20
    GROUP BY bid_count ORDER BY bid_count
""").pl()

fig9 = px.bar(bids.to_pandas(), x='bid_count', y='contract_count',
             title='Figure 9: Competition Level - Contract Count by Bids',
             color='contract_count', color_continuous_scale='Viridis')
fig9.update_layout(height=700)
fig9.show()
save_fig(fig9, '09_bid_distribution', 'Bid Distribution')

In [ ]:
# FIGURE 10: CPV Coverage
cpv_coverage = duckdb.sql(f"""
    SELECT SUBSTRING(tender_cpvs, 1, 2) as cpv_div, COUNT(*) as n_records
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_cpvs IS NOT NULL
    GROUP BY cpv_div ORDER BY n_records DESC LIMIT 20
""").pl()

fig10 = px.bar(cpv_coverage.to_pandas(), x='cpv_div', y='n_records',
              title='Figure 10: CPV Coverage - Top 20 Procurement Categories',
              color='n_records', color_continuous_scale='Plasma')
fig10.update_layout(height=700, xaxis_tickangle=-45)
fig10.show()
save_fig(fig10, '10_cpv_top_divisions', 'CPV Coverage')

In [ ]:
# FIGURE 11: Price Distribution
prices = duckdb.sql(f"""
    SELECT LOG(tender_finalprice + 1) as log_price
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_finalprice > 0 LIMIT 500000
""").pl()

fig11 = px.histogram(prices.to_pandas(), x='log_price', nbins=100,
                    title='Figure 11: Contract Value Distribution (Log Scale)',
                    color_discrete_sequence=['#2ecc71'])
fig11.update_layout(height=700)
fig11.show()
save_fig(fig11, '11_price_distribution_log', 'Price Distribution')

In [ ]:
# FIGURE 12: Amendments
amendments = duckdb.sql(f"""
    SELECT tender_proceduretype as procedure, COUNT(*) as total,
           SUM(CASE WHEN contract_amendments > 0 THEN 1 ELSE 0 END) as amended,
           100.0 * SUM(CASE WHEN contract_amendments > 0 THEN 1 ELSE 0 END) / COUNT(*) as amendment_pct
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_proceduretype IS NOT NULL
    GROUP BY procedure ORDER BY amendment_pct DESC
""").pl()

fig12 = px.bar(amendments.to_pandas(), x='procedure', y='amendment_pct',
              title='Figure 12: Contract Amendment Rate by Procedure',
              color='amendment_pct', color_continuous_scale='RdYlGn_r')
fig12.update_layout(height=700, xaxis_tickangle=-45)
fig12.show()
save_fig(fig12, '12_amendment_rate', 'Amendment Rate')

In [ ]:
# FIGURE 13: Buyer Types
buyer_types = duckdb.sql(f"""
    SELECT tender_buyertype as buyer_type, COUNT(*) as count
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_buyertype IS NOT NULL
    GROUP BY buyer_type ORDER BY count DESC
""").pl()

fig13 = px.pie(buyer_types.to_pandas(), values='count', names='buyer_type',
              title='Figure 13: Buyer Ecosystem - Distribution by Type')
fig13.update_layout(height=700)
fig13.show()
save_fig(fig13, '13_buyer_type_distribution', 'Buyer Type Distribution')

In [ ]:
# FIGURE 14: HHI Trend
hhi_by_year = duckdb.sql(f"""
    WITH supplier_share AS (
        SELECT YEAR(tender_awarddecisiondate) as year, tender_suppliersme_id as supplier_id,
               SUM(COALESCE(tender_finalprice, 0)) as value
        FROM read_parquet('{PARQUET_PATH}')
        WHERE tender_awarddecisiondate IS NOT NULL AND tender_finalprice > 0
        GROUP BY year, supplier_id
    ),
    supplier_pct AS (
        SELECT year, supplier_id, value / SUM(value) OVER (PARTITION BY year) as share
        FROM supplier_share
    )
    SELECT year, SUM(share * share) * 10000 as hhi
    FROM supplier_pct GROUP BY year ORDER BY year
""").pl()

fig14 = px.line(hhi_by_year.to_pandas(), x='year', y='hhi', markers=True,
               title='Figure 14: Market Concentration (HHI) Trend',
               line_shape='spline')
fig14.update_layout(height=700)
fig14.add_hline(y=1500, line_dash='dash', line_color='red')
fig14.show()
save_fig(fig14, '14_hhi_trend', 'HHI Trend')

In [ ]:
# FIGURE 15: Summary Statistics
summary_stats = duckdb.sql(f"""
    SELECT COUNT(*) as total_contracts,
           COUNT(DISTINCT tender_buyersme_id) as unique_buyers,
           COUNT(DISTINCT tender_suppliersme_id) as unique_suppliers,
           ROUND(100.0 * SUM(CASE WHEN lot_bidscount = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) as single_bidder_pct,
           COUNT(DISTINCT SUBSTRING(tender_cpvs, 1, 2)) as cpv_divisions,
           ROUND(MEDIAN(tender_finalprice), 0) as median_price,
           ROUND(100.0 * SUM(CASE WHEN contract_amendments > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) as amendment_pct
    FROM read_parquet('{PARQUET_PATH}')
    WHERE tender_awarddecisiondate IS NOT NULL
""").pl()

stats = summary_stats.to_dicts()[0]
summary_data = {
    'Metric': ['Total Contracts', 'Unique Buyers', 'Unique Suppliers', 'Single-Bidder %', 'CPV Divisions', 'Median Price (€)', 'Amendment %'],
    'Value': [f"{stats['total_contracts']:,}", f"{stats['unique_buyers']:,}", f"{stats['unique_suppliers']:,}", 
              f"{stats['single_bidder_pct']:.1f}%", f"{stats['cpv_divisions']:.0f}", f"{stats['median_price']:,.0f}", f"{stats['amendment_pct']:.1f}%"]
}

fig15 = go.Figure(data=[go.Table(
    header=dict(values=['<b>Metric</b>', '<b>Value</b>'], fill_color='#1F4E79', font=dict(color='white', size=12)),
    cells=dict(values=[summary_data['Metric'], summary_data['Value']], fill_color=['#E8F0F7', '#F0F0F0'], font=dict(size=11), height=30)
)])
fig15.update_layout(title='Figure 15: EDA Summary Statistics', height=700)
fig15.show()
save_fig(fig15, '15_summary_statistics', 'Summary Statistics')

In [ ]:
# COMPLETION VERIFICATION
import os

expected = ['01_schema_nulls.png', '02_temporal_coverage.png', '03_buyer_value_distribution.png',
           '04_buyer_dependency_distribution.png', '05_top_suppliers.png', '06_supplier_churn.png',
           '07_procedure_distribution.png', '08_single_bidder_by_procedure.png', '09_bid_distribution.png',
           '10_cpv_top_divisions.png', '11_price_distribution_log.png', '12_amendment_rate.png',
           '13_buyer_type_distribution.png', '14_hhi_trend.png', '15_summary_statistics.png']

existing = [f for f in os.listdir('reports/eda_outputs') if f.endswith('.png')]

print('\n' + '='*80)
print('✅ EDA PNG EXPORT COMPLETION')
print('='*80)
print(f'Expected: {len(expected)} | Created: {len(existing)}')
total_mb = sum(os.path.getsize(f'reports/eda_outputs/{f}') for f in existing) / (1024*1024)
print(f'Total Size: {total_mb:.1f}MB')
print(f'Output: reports/eda_outputs/')
print('✅ READY FOR POWERPOINT & THESIS')
print('='*80)